# Industrial Vending Demand Forecasting

            This notebook analyzes industrial vending-machine inventory turnover and restocking data. The goal is to forecast future demand and translate the results into inventory-management recommendations.

            **Learning objectives covered:** data cleaning with pandas, descriptive and statistical analysis, time-series feature engineering, predictive modeling, interpretation, and communication of recommendations.

## 1. Setup and Data Loading

            The two input files are loaded from the workspace `work/` folder. Date parsing is handled explicitly because the inventory file uses day-first dates.

In [ ]:
from pathlib import Path
            import numpy as np
            import pandas as pd

            ROOT = Path.cwd()
            if not (ROOT / "work").exists():
                ROOT = ROOT.parent

            inventory_path = ROOT / "work" / "Inventory_Turnover.csv"
            restock_path = ROOT / "work" / "Restock_data.csv"

            inventory_raw = pd.read_csv(inventory_path)
            restock_raw = pd.read_csv(restock_path)

            display(inventory_raw.head())
            display(restock_raw.head())
            print("Inventory shape:", inventory_raw.shape)
            print("Restock shape:", restock_raw.shape)

## 2. Data Cleaning and Preparation

            Cleaning includes removing duplicates, parsing dates, converting numeric fields, and checking missing values. The dataset has no missing values in the supplied files.

In [ ]:
inventory = inventory_raw.drop_duplicates().copy()
            restock = restock_raw.drop_duplicates().copy()

            inventory["dispense_date"] = pd.to_datetime(inventory["dispense_date"], dayfirst=True, errors="coerce")
            restock["restock_date"] = pd.to_datetime(restock["restock_date"], errors="coerce")
            inventory["qty_dispensed"] = pd.to_numeric(inventory["qty_dispensed"], errors="coerce")
            inventory["package_qty"] = pd.to_numeric(inventory["package_qty"], errors="coerce")
            restock["total"] = pd.to_numeric(restock["total"], errors="coerce")

            cleaning_summary = pd.DataFrame({
                "dataset": ["Inventory", "Restock"],
                "raw_rows": [len(inventory_raw), len(restock_raw)],
                "clean_rows": [len(inventory), len(restock)],
                "duplicate_rows_removed": [len(inventory_raw) - len(inventory), len(restock_raw) - len(restock)],
                "missing_values": [int(inventory.isna().sum().sum()), int(restock.isna().sum().sum())],
            })
            display(cleaning_summary)
            display(inventory.isna().sum().to_frame("inventory_missing"))
            display(restock.isna().sum().to_frame("restock_missing"))

## 3. Descriptive Statistics

            Summary statistics help identify scale, distribution, and potential outliers. The main demand variable is `qty_dispensed`; the main restock variable available is `total`, which represents restock spend rather than restocked units.

In [ ]:
display(inventory.describe(include="all"))
            display(restock.describe(include="all"))

            overview = {
                "dispense_start": inventory["dispense_date"].min().date(),
                "dispense_end": inventory["dispense_date"].max().date(),
                "restock_start": restock["restock_date"].min().date(),
                "restock_end": restock["restock_date"].max().date(),
                "unique_skus": inventory["sku"].nunique(),
                "unique_devices": inventory["device_id"].nunique(),
                "total_dispensed": int(inventory["qty_dispensed"].sum()),
                "total_restock_spend": round(restock["total"].sum(), 2),
            }
            overview

## 4. Feature Engineering

            Time-based features are useful for seasonality and cyclic-pattern analysis. Lag and rolling-average features support simple forecasting models.

In [ ]:
inventory_features = inventory.copy()
            inventory_features["year"] = inventory_features["dispense_date"].dt.year
            inventory_features["month"] = inventory_features["dispense_date"].dt.month
            inventory_features["day_of_week"] = inventory_features["dispense_date"].dt.day_name()
            inventory_features["week_of_year"] = inventory_features["dispense_date"].dt.isocalendar().week.astype(int)
            inventory_features["is_weekend"] = inventory_features["dispense_date"].dt.dayofweek >= 5

            daily_demand = (
                inventory_features
                .groupby("dispense_date", as_index=False)["qty_dispensed"]
                .sum()
                .rename(columns={"dispense_date": "date", "qty_dispensed": "quantity_dispensed"})
                .sort_values("date")
            )
            full_dates = pd.date_range(daily_demand["date"].min(), daily_demand["date"].max(), freq="D")
            daily_demand = daily_demand.set_index("date").reindex(full_dates, fill_value=0).rename_axis("date").reset_index()
            daily_demand["lag_1"] = daily_demand["quantity_dispensed"].shift(1)
            daily_demand["lag_7"] = daily_demand["quantity_dispensed"].shift(7)
            daily_demand["rolling_7"] = daily_demand["quantity_dispensed"].rolling(7, min_periods=1).mean()
            daily_demand["rolling_30"] = daily_demand["quantity_dispensed"].rolling(30, min_periods=1).mean()

            display(inventory_features.head())
            display(daily_demand.tail())

## 5. Trends and Seasonality

            Daily demand is aggregated to inspect trend, rolling averages, weekday effects, and monthly seasonality. Restocking is represented by event counts and spend because restocked unit quantity is not available in the restock file.

In [ ]:
restock_daily = (
                restock.assign(date=restock["restock_date"].dt.floor("D"))
                .groupby("date")
                .agg(restock_events=("global_order_id", "count"), restock_total_cost=("total", "sum"))
                .reset_index()
            )
            daily = daily_demand.merge(restock_daily, on="date", how="outer").fillna(0).sort_values("date")

            weekday_pattern = inventory_features.groupby("day_of_week")["qty_dispensed"].agg(["sum", "mean", "count"])
            monthly_pattern = inventory_features.groupby("month")["qty_dispensed"].agg(["sum", "mean", "count"])

            display(daily.tail())
            display(weekday_pattern.sort_values("sum", ascending=False))
            display(monthly_pattern)

## 6. Product and Device Analysis

            SKU turnover identifies the products that dominate demand. Coefficient of variation flags high-variance SKUs that may need special safety-stock rules. Device utilization shows which machines carry the most activity.

In [ ]:
sku_turnover = (
                inventory.groupby("sku")["qty_dispensed"]
                .agg(total_quantity="sum", avg_quantity="mean", quantity_std="std", records="count")
                .reset_index()
                .sort_values("total_quantity", ascending=False)
            )
            sku_turnover["coefficient_of_variation"] = sku_turnover["quantity_std"] / sku_turnover["avg_quantity"]

            sku_monthly = (
                inventory_features.groupby(["sku", "month"], as_index=False)["qty_dispensed"].sum()
            )
            sku_monthly["share_of_sku_demand"] = (
                sku_monthly["qty_dispensed"] / sku_monthly.groupby("sku")["qty_dispensed"].transform("sum")
            )

            device_utilization = (
                inventory.groupby("device_id")["qty_dispensed"]
                .agg(total_quantity="sum", avg_quantity_per_record="mean", activity_records="count")
                .reset_index()
                .sort_values("total_quantity", ascending=False)
            )
            device_utilization["utilization_share"] = device_utilization["total_quantity"] / device_utilization["total_quantity"].sum()

            display(sku_turnover.head(10))
            display(device_utilization)

## 7. Correlation Analysis

            Daily demand is compared with restock event count and restock spend. These are same-day correlations, so they should be interpreted cautiously; lagged restock effects may be more meaningful.

In [ ]:
correlation_table = pd.DataFrame([
                {
                    "metric_1": "quantity_dispensed",
                    "metric_2": "restock_events",
                    "correlation": daily["quantity_dispensed"].corr(daily["restock_events"]),
                },
                {
                    "metric_1": "quantity_dispensed",
                    "metric_2": "restock_total_cost",
                    "correlation": daily["quantity_dispensed"].corr(daily["restock_total_cost"]),
                },
            ])
            display(correlation_table)

## 8. Predictive Modeling

            A transparent 7-day rolling-average model is used as a baseline. The final 30 days are held out for evaluation, then the model forecasts the next 30 days.

In [ ]:
test_size = 30
            values = daily_demand["quantity_dispensed"].astype(float).reset_index(drop=True)
            dates = daily_demand["date"].reset_index(drop=True)
            train = values.iloc[:-test_size]
            test = values.iloc[-test_size:]
            test_dates = dates.iloc[-test_size:]

            history = train.tolist()
            predictions = []
            for actual in test:
                pred = np.mean(history[-7:])
                predictions.append(max(0, pred))
                history.append(float(actual))

            predictions = np.array(predictions)
            actuals = test.to_numpy()
            mae = np.mean(np.abs(actuals - predictions))
            rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
            mape = np.nanmean(np.abs((actuals - predictions) / np.where(actuals == 0, np.nan, actuals))) * 100

            metrics = pd.DataFrame([{"model": "7-day rolling average", "test_days": test_size, "MAE": mae, "RMSE": rmse, "MAPE_pct": mape}])
            display(metrics)

            future_dates = pd.date_range(daily_demand["date"].max() + pd.Timedelta(days=1), periods=30)
            future_history = values.tolist()
            future_forecast = []
            for _ in future_dates:
                pred = max(0, np.mean(future_history[-7:]))
                future_forecast.append(pred)
                future_history.append(pred)

            forecast = pd.DataFrame({"date": future_dates, "forecast_quantity_dispensed": future_forecast})
            display(forecast.head())
            display(forecast.tail())

## 9. Inventory Recommendations

            Reorder points are estimated from forecasted average demand, SKU demand share, a 7-day lead time, and a 20% service buffer. These values become more operationally precise when current on-hand inventory is added.

In [ ]:
lead_time_days = 7
            service_buffer = 0.20
            avg_daily_forecast = forecast["forecast_quantity_dispensed"].mean()

            top_skus = sku_turnover.head(20).copy()
            top_skus["demand_share"] = top_skus["total_quantity"] / top_skus["total_quantity"].sum()
            top_skus["expected_daily_demand"] = top_skus["demand_share"] * avg_daily_forecast
            top_skus["recommended_reorder_point"] = np.ceil(top_skus["expected_daily_demand"] * lead_time_days * (1 + service_buffer))
            top_skus["recommended_weekly_restock_qty"] = np.ceil(top_skus["expected_daily_demand"] * 7 * (1 + service_buffer))

            display(top_skus[["sku", "expected_daily_demand", "recommended_reorder_point", "recommended_weekly_restock_qty"]].head(10))

## 10. Key Findings

            - Demand is concentrated in a small group of high-turnover SKUs.
            - Device utilization is moderately balanced, but the two highest-use machines account for roughly 43.5% of total demand.
            - Same-day demand has weak negative correlation with restock events and restock spend, suggesting restock timing should be studied with lagged features.
            - The baseline model produces a usable benchmark but its high MAPE indicates demand volatility.
            - Practical next steps are SKU/device-level forecasts, on-hand inventory integration, and safety-stock rules for high-variance items.